# 【講師用】Ch.5 総合演習 完全解説・模範解答

> 🔒 **このノートブックは講師専用です。受講者には配布しないでください。**

---

## 📢 講師向け冒頭ガイド

### この演習の位置づけ
- Ch.1〜4 で学んだ全スキルを「自力で」使う演習
- 新しいデータ（乳がん診断）で「手順を覚えているか」ではなく「判断できるか」を試す
- 正解は1つではない。EDAの仮説や考察の質を評価する

### 進行上のポイント
| 時間 | 対応 |
|------|------|
| 最初の5分 | データの概要を口頭で説明。医療データの重みを伝える |
| 演習中（随時） | 詰まっている受講者には「まずどのChapterの何をやれば良いか」だけヒントを出す |
| 70分後 | Step 4 の考察を書かせる。コードよりも言語化を優先 |
| 最後10分 | 3〜5名に発表してもらう。特に「Recall を重視すべき理由」を引き出す |

### よくある躓きパターン
| 躓き | 対処法 |
|------|--------|
| DataFrame に変換できない | `cancer.data` と `cancer.feature_names` を使うことを確認 |
| target 列の意味がわからない | 「0=悪性、1=良性と医師が判断したデータ」と説明 |
| Step3 でエラーが出る | `X` と `y` を定義できているか・スケーリングを train にだけ fit しているかを確認 |

### この演習特有の注意点
- **クラスの偏り**：悪性212件 / 良性357件（約37%:63%）
  → 正解率が高くても Recall（悪性の検出率）が低い場合がある
- **医療の文脈での誤分類のリスク**
  - 悪性を良性と誤判定（FN）= 見逃し → 最も危険
  - 良性を悪性と誤判定（FP）= 過剰検査 → 患者の心理的・経済的負担

---
## Step 1：データ読み込みと基本確認（模範解答）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# データの読み込み
cancer = load_breast_cancer()

# DataFrame に変換
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['diagnosis'] = cancer.target  # 0=悪性(malignant), 1=良性(benign)
df['diagnosis_label'] = df['diagnosis'].map({0: '悪性(malignant)', 1: '良性(benign)'})

print('=== データセット概要 ===')
print(f'件数: {len(df)} 件')
print(f'特徴量数: {len(cancer.feature_names)} 列')

In [ ]:
# 先頭5行を確認
df.head()

In [ ]:
# 基本統計量
print('=== 基本統計量（先頭5列）===')
df[cancer.feature_names[:5]].describe().round(2)

In [ ]:
# 欠損値の確認
print('=== 欠損値の件数 ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '欠損値はありません ✅')

In [ ]:
# クラスの分布（重要！）
print('=== 悪性・良性の件数と割合 ===')
print(df['diagnosis_label'].value_counts())
print()
print('割合（%）：')
print((df['diagnosis_label'].value_counts(normalize=True) * 100).round(1))

### 📢 Step 1 の解説ポイント

**欠損値について：**
> 実際のデータには欠損値がありません。これは整備されたベンチマークデータセットのため。
> 実務データには必ず欠損があります。「欠損値がない」というのは珍しいことです。

**クラスの偏りについて（重要）：**
> 悪性37.3% / 良性62.7%。完全に均等ではありませんが、極端な不均衡でもありません。
> ただし「全部良性と予測するモデル」でも正解率が62.7%になります。
> → **Recall（悪性の見逃しがないか）に注目させる**

**Step 1 の気づき例（講師が板書 or 口頭で示す）：**
```
- データ件数：569件（ワインの178件より多い）
- 欠損値：なし（ベンチマークデータのため）
- クラスの偏り：悪性37% / 良性63%（軽度の偏りあり）
  → Accuracy だけでなく Recall を必ず確認する必要がある
```

---
## Step 2：EDA（可視化・相関確認）（模範解答）

In [ ]:
# ─── グラフ1：悪性・良性で「mean radius（平均半径）」を箱ひげ図で比較 ────
# 「腫瘍の大きさ（半径）」は悪性・良性で差があるか？

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 箱ひげ図
for i, col in enumerate(['mean radius', 'mean texture']):
    data_m = df[df['diagnosis'] == 0][col]  # 悪性
    data_b = df[df['diagnosis'] == 1][col]  # 良性
    axes[i].boxplot([data_m, data_b], labels=['悪性(0)', '良性(1)'],
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'{col}\n（悪性 vs 良性）')
    axes[i].set_ylabel('値')

plt.suptitle('特徴量の悪性・良性比較（箱ひげ図）', fontsize=13)
plt.tight_layout()
plt.show()

### 📢 グラフ1の解説
- `mean radius`（腫瘍の平均半径）：**悪性の方が明らかに大きい**
  → 腫瘍が大きいほど悪性の可能性が高いというのは医学的にも自然
- `mean texture`：悪性の方が値が高い傾向（分布の重なりあり）
> 「箱が重なっていない特徴量ほど、悪性・良性の分類に効くと考えられます。」

In [ ]:
# ─── グラフ2：mean radius × mean concave points の散布図 ─────────
# 2特徴量の組み合わせで悪性・良性が分離できるか？

fig, ax = plt.subplots(figsize=(8, 6))

for label, name, color in [(0, '悪性(malignant)', '#e74c3c'),
                            (1, '良性(benign)',    '#2ecc71')]:
    subset = df[df['diagnosis'] == label]
    ax.scatter(subset['mean radius'], subset['mean concave points'],
               label=name, color=color, alpha=0.6, s=50)

ax.set_title('mean radius × mean concave points\n（悪性 vs 良性）')
ax.set_xlabel('mean radius（平均半径）')
ax.set_ylabel('mean concave points（平均凹点数）')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─── 相関ヒートマップ（30特徴量 + ターゲット）────────────────────
# ターゲット（diagnosis）との相関が高い特徴量を特定する

# ターゲットとの相関が高い上位10特徴量を選ぶ（30列全部は見にくいため）
corr_with_target = df[cancer.feature_names].corrwith(df['diagnosis']).abs().sort_values(ascending=False)
top10_features = corr_with_target.head(10).index.tolist()

print('ターゲット（diagnosis）との相関が高い特徴量 Top10：')
print(corr_with_target.head(10).round(3))

In [ ]:
# 上位10特徴量 + target でヒートマップ
cols_for_heatmap = top10_features + ['diagnosis']
corr_matrix = df[cols_for_heatmap].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix,
            annot=True, fmt='.2f',
            cmap='coolwarm', center=0,
            square=True, ax=ax)
ax.set_title('主要特徴量とターゲットの相関係数（上位10特徴量）')
plt.tight_layout()
plt.show()

### 📢 EDA の解説スクリプト

```
「ターゲット（diagnosis）との相関が高い上位特徴量を見ると：

  diagnosis との相関が高い = 負の相関
  （diagnosis は 0=悪性, 1=良性 なので、
   値が高い = 悪性の特徴 の場合は diagnosis と負の相関）

  絶対値が大きい特徴量（mean concave points, mean radius 等）が
  悪性・良性の分類に効きそう → これを仮説として Ch.4 で検証します。」
```

**EDAの仮説（模範例）：**
```
「mean concave points（腫瘍の凹点数の平均）が diagnosis と最も相関が高い。
  腫瘍の形状が不規則（凹みが多い）な場合に悪性である可能性が高いと予想する。
  worst radius（腫瘍の最大半径）も重要だと思う。
  なぜなら、最大半径が大きいほど悪性の可能性が高いと直感的にも考えられるから。」
```

---
## Step 3：モデル構築・評価（模範解答）

In [ ]:
# 特徴量と ターゲットを定義
X = df[cancer.feature_names]  # 全30特徴量
y = df['diagnosis']            # 0=悪性, 1=良性

# データ分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y       # クラスの比率を保つ
)

print('訓練データ:', X_train.shape)
print('テストデータ:', X_test.shape)

# StandardScaler でスケーリング（fit は訓練データのみ！）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ランダムフォレストモデルの学習
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)
print('\n学習完了！')

In [ ]:
# 評価
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

print(f'正解率（Accuracy）: {acc:.1%}')
print()

# 混同行列
cm = confusion_matrix(y_test, y_pred)
class_names = ['悪性(0)', '良性(1)']

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm,
            annot=True, fmt='d',
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            linewidths=1)
ax.set_title('混同行列（乳がん診断）', fontsize=13)
ax.set_xlabel('予測ラベル', fontsize=11)
ax.set_ylabel('正解ラベル', fontsize=11)
plt.tight_layout()
plt.show()

# 詳細レポート
print('=== 評価レポート ===')
print(classification_report(y_test, y_pred, target_names=class_names))

### 📢 混同行列の詳細解説（医療文脈）

```
混同行列の各マスの意味（医療文脈）：

┌────────────────┬──────────────┬──────────────┐
│                │  予測:悪性   │  予測:良性   │
├────────────────┼──────────────┼──────────────┤
│ 実際:悪性      │ TP（正検出） │ FN（見逃し）  │
│                │ ✅ 正しく    │ ⚠️ 悪性を    │
│                │    悪性と判定│    良性と誤判定│
├────────────────┼──────────────┼──────────────┤
│ 実際:良性      │ FP（誤検知） │ TN（正検出） │
│                │ ⚠️ 良性を   │ ✅ 正しく    │
│                │    悪性と誤判定│   良性と判定 │
└────────────────┴──────────────┴──────────────┘

最も危険：FN（悪性を良性と見逃す）
  → 治療が遅れ、患者の生命に関わる
  → Recall（再現率）= TP / (TP + FN) を最大化したい

次に問題：FP（良性を悪性と誤検知）
  → 不必要な生検・手術・精神的ストレス
  → Precision（適合率）も重要
```

In [ ]:
# 特徴量重要度
importances = pd.Series(model.feature_importances_, index=cancer.feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 10))
colors_list = ['#e74c3c' if imp > 0.05 else 'steelblue' for imp in importances.values]
importances.plot(kind='barh', ax=ax, color=colors_list)
ax.axvline(x=1/30, color='gray', linestyle='--', alpha=0.7,
           label=f'均等な場合（{1/30:.3f}）')
ax.set_title('特徴量重要度（乳がん診断）', fontsize=14)
ax.set_xlabel('重要度')
ax.legend()
plt.tight_layout()
plt.show()

print('重要度トップ5：')
print(importances.sort_values(ascending=False).head(5).round(4))

### 📢 特徴量重要度と EDA の仮説対照

```
典型的な結果：
  worst concave points が最も重要 → EDA での「mean concave points が重要そう」
  worst radius, worst perimeter なども上位

「worst_***」特徴量が多い：
  worst = 「最も悪い値（最大値）」を意味する
  腫瘍の最大サイズ・最も不規則な形状 が悪性判定に効くのは医学的にも自然

EDAの仮説との一致：
  「concave points が重要」という仮説が検証された
```

---
## Step 4：考察のまとめ（模範解答・発表ガイド）

### 4-1. データからわかること（模範解答）

```
1. 乳がんの悪性・良性は、腫瘍の形状（凹点数・不規則性）と大きさ（半径・面積）
   の組み合わせで高精度（95%以上）に判別できる。

2. 特に「worst（最大値）系」の特徴量が重要で、
   一番大きな計測値が診断の決め手になる傾向がある。

3. ただしモデルに数件の見逃し（FN）があり、
   医療現場での実用化には Recall のさらなる向上と
   人間の医師による確認が必要。
```

### 4-2. 実務で使う場合の注意点（模範解答）

```
最大の注意点：見逃し（FN）のリスク

悪性を良性と誤判定（FN）:
  → 患者が治療を受けられない → 病気が進行 → 最悪、死につながる
  → 絶対に最小化したい

良性を悪性と誤判定（FP）:
  → 不要な検査・手術 → 心理的・金銭的負担
  → 問題だが FN よりは影響が小さい

対策：
  1. 閾値を下げる（デフォルトは0.5 → 0.3に下げると Recall が上がるが Precision が下がる）
  2. モデルだけで判断しない → 医師の最終確認を必須にする（Human-in-the-loop）
  3. 定期的に再学習する（患者の傾向は時代で変わる可能性がある）
```

In [ ]:
# 📢 発展：閾値を変えて Precision と Recall のトレードオフを確認
from sklearn.metrics import precision_score, recall_score

# 各サンプルの「悪性（クラス0）である確率」を取得
y_prob_malignant = model.predict_proba(X_test_scaled)[:, 0]  # クラス0の確率

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
print('閾値を変えた場合のトレードオフ：')
print(f'{"閾値":>6} | {"Precision（悪性）":>17} | {"Recall（悪性）":>15} | {"Accuracy":>10}')
print('-' * 58)

for thr in thresholds:
    # 確率 >= thr なら悪性（0）と予測
    y_pred_thr = (y_prob_malignant >= thr).astype(int)
    y_pred_thr = 1 - y_pred_thr  # 0=悪性の予測を反転（1-で良性予測に変換）
    # 悪性（label=0）を陽性として計算
    prec = precision_score(y_test, y_pred_thr, pos_label=0, zero_division=0)
    rec  = recall_score(y_test, y_pred_thr, pos_label=0, zero_division=0)
    acc_thr = accuracy_score(y_test, y_pred_thr)
    print(f'{thr:>6.1f} | {prec:>17.1%} | {rec:>15.1%} | {acc_thr:>10.1%}')

### 📢 閾値トレードオフの解説スクリプト

```
「閾値を下げると（0.5 → 0.3）：
  Recall が上がる = 悪性を見逃しにくくなる
  Precision が下がる = 良性を悪性と誤判定しやすくなる

どの閾値を選ぶかは「ビジネス要件」次第です。

医療（がん検診）の場合：
  見逃し（FN）のリスク > 過剰検査（FP）のリスク
  → 閾値を低め（例：0.3）に設定して Recall を優先する判断もある

この判断はデータサイエンティストではなく、
医師・病院経営者・倫理委員会が行います。
DSの役割は「トレードオフを可視化して提示する」こと。」
```

### 4-3. 生成AI活用の振り返り（発表時の引き出し質問）

**受講者に聞く質問例：**

```
Q1: AI を使ってよかった場面はどんな場面でしたか？
    → 「confusion_matrix の色の変え方を調べた」
       「plt.subplots の書き方を忘れていた」など

Q2: AI に頼みすぎて困った場面はありましたか？
    → 「全部のコードを生成させたら何をしているか分からなくなった」など

Q3: AI を使っていても「自分で判断しなければいけなかった」場面は？
    → 「どの特徴量を可視化するか」
       「Recall を重視すべきかどうか」
       「EDAで立てた仮説の内容」
```

### 4-4. 発展：モデルの精度をさらに上げるには（解答例）

In [ ]:
# 発展案1：重要度上位の特徴量だけでモデルを再構築
top_n = 10
top_features = importances.sort_values(ascending=False).head(top_n).index.tolist()

X_train_top = X_train[top_features]
X_test_top  = X_test[top_features]

scaler_top = StandardScaler()
X_train_top_scaled = scaler_top.fit_transform(X_train_top)
X_test_top_scaled  = scaler_top.transform(X_test_top)

model_top = RandomForestClassifier(n_estimators=100, random_state=42)
model_top.fit(X_train_top_scaled, y_train)
y_pred_top = model_top.predict(X_test_top_scaled)

print(f'全30特徴量:   Accuracy = {acc:.1%}')
print(f'上位{top_n}特徴量: Accuracy = {accuracy_score(y_test, y_pred_top):.1%}')
print()
print(f'=== 上位{top_n}特徴量モデルの詳細評価 ===')
print(classification_report(y_test, y_pred_top, target_names=['悪性(0)', '良性(1)']))

---
## 📢 発表進行ガイド（最後10分）

### 発表者への質問リスト（進行に使ってください）

```
【EDA について】
- どの特徴量に注目しましたか？なぜですか？
- グラフから「悪性らしさ」はどんな特徴だと思いましたか？

【モデル評価について】
- 正解率は何%でしたか？
- 混同行列で見逃し（FN）は何件ありましたか？
- 医師の立場なら、この結果で診断補助に使えると思いますか？

【AI活用について】
- 今日の演習で AI をどんな場面で使いましたか？
- コードを AI が生成したとき、何をやっているか理解できましたか？
```

### まとめのメッセージ（クロージングで伝える）

```
今日1日で体験したことは、実際の DS の仕事の縮小版です。

コードを書く技術は AI が補ってくれますが：
  - どのデータを分析するかを決めるのは「あなた」
  - グラフから仮説を立てるのは「あなた」
  - 数値を解釈してビジネスの言葉にするのは「あなた」
  - Recall と Precision のどちらを優先するかを判断するのは「あなた」

これが「DS と AI の協業」の姿です。
```

---
## ✅ 1日のまとめ（講師用チェックリスト）

### 受講者が習得できたか確認するポイント

| チャプター | 確認ポイント | チェック |
|-----------|------------|--------|
| Ch.1 | `groupby` の使い方と結果の解釈ができる | ☐ |
| Ch.2 | グラフから仮説を言語化できる | ☐ |
| Ch.3 | 「画像 = 数値の配列」を説明できる | ☐ |
| Ch.4 | `scaler.fit` は訓練データのみ、と理由を説明できる | ☐ |
| Ch.4 | 混同行列の各マスの意味を説明できる | ☐ |
| Ch.5 | Recall を優先すべき理由を医療の文脈で説明できる | ☐ |
| 全体 | AI に任せてよいことと自分でやることを区別できている | ☐ |